# Module 2: NLP & Transformers for Educational Question Understanding
## AI Study Assistant - From Scratch

Welcome to **Module 2**! Is notebook mein hum samjhenge ki computer natural language (English/Hinglish questions) ko kaise decode aur mathematically represent karta hai.

---

### Core Concepts in Simple Hinglish:

1. **Text Preprocessing & Normalization**:
   - Raw text me typos, casing variations, HTML tags, aur special punctuation hote hain.
   - Normalization text ko standard uniform structure me lata hai.

2. **Stemming vs. Lemmatization**:
   - **Stemming (e.g. Porter Stemmer)**: Suffixes ko rule-based tarike se cut karta hai. Fast hota hai lekin aksar invalid non-dictionary words produce karta hai (jaise `studying` -> `studi`, `universal` -> `univers`).
   - **Lemmatization**: Dictionary aur morphological analysis (POS tagging) ka use karke word ke actual base form (lemma) tak pahunchta hai (jaise `studies` -> `study`, `better` -> `good`, `matrices` -> `matrix`).

3. **Tokenization (Word vs Subword)**:
   - **Word-level**: Spaces se split karna (vocabulary explosion aur Out-Of-Vocabulary words ki problem).
   - **Subword Tokenization (BPE / WordPiece)**: Words ko meaningful chunks me todna (jaise `unsupervised` -> `un` + `supervised`). Ye modern LLMs ka foundation hai.

4. **Dense Embeddings**:
   - Pehle sparse vectors (One-Hot, TF-IDF) use hote the jinme semantic meaning capture nahi hoti thi.
   - Dense Embeddings (jaise 384 dimensions) continuous vector space me words aur sentences ka contextual meaning capture karte hain. Similar meaning wale questions paas-paas hote hain.

5. **Self-Attention Mechanism**:
   - Har token baaki sabhi tokens ke sath interact karta hai aur attention score compute karta hai.
   - Formula:
     $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$
   - **Query (Q)**: "Main kya dhoondh raha hoon?"
   - **Key (K)**: "Mere paas kya information hai?"
   - **Value (V)**: "Mera actual content kya hai?"

6. **Transformer Architecture (Vaswani et al., 2017)**:
   - RNNs/LSTMs sequential the (slow training, vanishing gradients).
   - Transformers pure sentence ko parallel me process karte hain.
   - Positional Encodings token order ka sense maintain rakhte hain.
   - Multi-Head Attention ek hi sentence ko alag-alag angles (syntax, semantics, context) se observe karta hai.

## Step 1: Stemming vs. Lemmatization Demo

In [ ]:
import sys
sys.path.insert(0, "..")

from src.nlp.preprocessor import NLPPreprocessor

preprocessor = NLPPreprocessor()
sample_words = ["studying", "studies", "matrices", "regularization", "algorithms", "better", "running"]
comparison = preprocessor.compare_stem_vs_lemma(sample_words)

print(f"{'Original Word':<18} | {'Stemmed (Cut)':<15} | {'Lemmatized (Valid)':<18}")
print("-" * 58)
for row in comparison:
    print(f"{row['original']:<18} | {row['stemmed']:<15} | {row['lemmatized']:<18}")

## Step 2: Step-by-Step Scaled Dot-Product Attention
Dekhte hain ki PyTorch me Query, Key, aur Value se attention weights kaise bante hain.

In [ ]:
import torch
from src.nlp.transformer_pipeline import compute_scaled_dot_product_attention

# Simulating 3 tokens (e.g. ['python', 'yields', 'generators']) with d_k = 4
seq_len = 3
d_k = 4

Q = torch.randn(1, 1, seq_len, d_k)
K = torch.randn(1, 1, seq_len, d_k)
V = torch.randn(1, 1, seq_len, d_k)

context, weights = compute_scaled_dot_product_attention(Q, K, V)

print("Attention Weights Matrix (Sum across rows = 1.0):")
print(weights.squeeze())
print("\nRow sums:", weights.squeeze().sum(dim=-1))
print("\nOutput Context Vectors Shape:", context.shape)

## Step 3: Hugging Face Pretrained Transformer (all-MiniLM-L6-v2)
Ab hum real pretrained Transformer se dense semantic embeddings aur natural language question understanding perform karenge.

In [ ]:
from src.nlp.transformer_pipeline import TransformerQuestionUnderstander

understander = TransformerQuestionUnderstander()
analysis = understander.understand_question("What is the difference between list comprehensions and generators in Python?")

for k, v in analysis.items():
    print(f"{k:<25}: {v}")

## Step 4: Semantic Similarity & Paraphrase Detection
Student question ko alag-alag tarike se puchh sakta hai (e.g. 'How does gradient descent work?' vs 'Explain gradient based optimization in ML').
Transformer models words match karne ki jagah meaning match karte hain!

In [ ]:
q1 = "How does gradient descent optimize weights?"
q2 = "Explain weight updates using gradients in machine learning."
q3 = "How do I write a decorator in Python?"

sim_1_2 = understander.compute_similarity(q1, q2)
sim_1_3 = understander.compute_similarity(q1, q3)

print(f"Similarity between Q1 and Q2 (Same ML topic, different words): {sim_1_2:.4f}")
print(f"Similarity between Q1 and Q3 (Completely different topics)     : {sim_1_3:.4f}")

candidates = [
    "What is stochastic gradient descent and learning rate?",
    "Explain Python list append versus extend.",
    "What is A* search algorithm in artificial intelligence?",
    "How does L2 ridge regularization prevent overfitting?"
]

print("\nTop similar questions for 'Tell me about gradient descent':")
results = understander.find_most_similar("Tell me about gradient descent", candidates, top_k=2)
for r in results:
    print(f"  Rank {r['rank']}: {r['candidate']} (Score: {r['similarity']})")